In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd

In [2]:
def solution_for_h2(L1, L2, R1, c1, R2, c2, P, mu, n, h2):
    n_star = n

    if n_star >= 1:
        def build_qbd_blocks_queue_only_i(L1, L2, h2, mu, n_star):
            lam1 = float(L1)
            lam2 = float(L2) * float(h2)
        
            m = int(n_star)  # phases i = 0..n_star-1 (queue-only)
        
            # A2: up in j
            A2 = lam2 * np.eye(m)
        
            # A0: down in j only from i=0
            A0 = np.zeros((m, m))
            A0[0, 0] = mu
        
            # A1: same level dynamics in i (plus diagonals incl. lam2 and (i=0) mu-down)
            A1 = np.zeros((m, m))
        
            # i -> i+1 (type1 arrival accepted) for i=0..m-2
            for i in range(0, m-1):
                A1[i, i+1] = lam1
        
            # i -> i-1 (service completes type1) for i=1..m-1
            for i in range(1, m):
                A1[i, i-1] = mu
        
            # diagonals (include all outflows: within-level + up + down(if i=0))
            for i in range(m):
                out = 0.0
                out += lam2  # up always
                if i <= m-2:
                    out += lam1  # type1 arrival accepted
                if i >= 1:
                    out += mu   # within-level service for type1
                else:
                    out += mu   # i=0: service is DOWN in j (A0)
                A1[i, i] = -out
        
            return A0, A1, A2
        
        def solve_R_iter(A0, A1, A2, tol=1e-12, max_iter=5000):
            # Iteration: R_{k+1} = -A2 (A1 + R_k A0)^{-1}
            m = A1.shape[0]
            R = np.zeros((m, m))
            for _ in range(max_iter):
                M = A1 + R @ A0
                R_next = np.linalg.solve(M.T, (-A2).T).T
                if np.max(np.abs(R_next - R)) < tol:
                    return R_next
                R = R_next
            raise RuntimeError("R did not converge (check stability / parameters).")
        
        def solve_boundary_and_pi1(L1, L2, h2, mu, n_star, tol=1e-12):
            # Boundary states: empty e plus level0 phases (i, j=0, k=1), i=0..m-1
            lam1 = float(L1)
            lam2 = float(L2) * float(h2)
            m = int(n_star)
            empty = m  # index in pi0
        
            A0, A1, A2 = build_qbd_blocks_queue_only_i(L1, L2, h2, mu, n_star)
            R = solve_R_iter(A0, A1, A2, tol=tol)
        
            m0 = m + 1
            B00 = np.zeros((m0, m0))
            B01 = np.zeros((m0, m))
            B10 = np.zeros((m, m0))
        
            # empty -> (i=0, j=0, k=1) with rate lam1+lam2
            B00[empty, 0] = lam1 + lam2
            # (i=0,j=0,k=1) -> empty with rate mu
            B00[0, empty] = mu
        
            # level0 i-dynamics (j=0,k=1): i->i+1 with lam1 for i=0..m-2; i->i-1 with mu
            for i in range(0, m-1):
                B00[i, i+1] = lam1
            for i in range(1, m):
                B00[i, i-1] = mu
        
            # level0 -> level1 via type2 arrival when busy: j 0->1
            for i in range(m):
                B01[i, i] = lam2
        
            # level1 -> level0: from (i=0,j=1) service completion goes to empty (because after down, j=0 and i=0 => empty)
            B10[0, empty] = mu
        
            # diagonals for B00 include out to level1
            for s in range(m0):
                out = np.sum(B00[s, :]) + np.sum(B01[s, :])
                B00[s, s] = -out
        
            # Solve for [pi0, pi1] with:
            # pi0 B00 + pi1 B10 = 0
            # pi0 B01 + pi1 (A1 + R A0) = 0
            # normalization: pi0*1 + pi1*(I-R)^{-1}*1 = 1
            dim = m0 + m
            G = np.zeros((dim, dim))
            G[:m0, :m0] = B00
            G[:m0, m0:] = B01
            G[m0:, :m0] = B10
            G[m0:, m0:] = A1 + R @ A0
        
            I = np.eye(m)
            ones0 = np.ones(m0)
            ones1 = np.ones(m)
            v = np.linalg.solve(I - R, ones1)  # (I-R)^{-1} 1
            norm_row = np.concatenate([ones0, v])
        
            M = G.T.copy()
            b = np.zeros(dim)
            M[-1, :] = norm_row
            b[-1] = 1.0
        
            x = np.linalg.solve(M, b)
            pi0 = x[:m0]  # includes empty at index 'empty'
            pi1 = x[m0:]  # phases at level j=1
            return pi0, pi1, R, empty
        
        def state_prob(i, j, k, pi0, pi1, R, n_star, empty_index):
            m = int(n_star)
            if k == 0:
                return float(pi0[empty_index]) if (i == 0 and j == 0) else 0.0
            if j == 0:
                return float(pi0[i]) if (0 <= i <= m-1) else 0.0
            # j>=1
            if not (0 <= i <= m-1):
                return 0.0
            pij = pi1
            for _ in range(j-1):
                pij = pij @ R
            return float(pij[i])
        
        import numpy as np
        
        def list_states_until_tail_mass(pi0, pi1, R, n_star, empty_index, eps=1e-12, Jmax_cap=1_000_000):
            m = int(n_star)
            out = {(0,0,0): float(pi0[empty_index])}
            for i in range(m):
                out[(i,0,1)] = float(pi0[i])
        
            I = np.eye(m)
            ones = np.ones(m)
        
            # total probability mass in levels j>=1
            total_ge1 = float(pi1 @ np.linalg.solve(I - R, ones))
        
            pij = pi1.copy()   # j=1
            mass_listed = 0.0
        
            for j in range(1, Jmax_cap + 1):
                level_mass = float(np.sum(pij))
                mass_listed += level_mass
        
                for i in range(m):
                    p = float(pij[i])
                    if p > 0.0:  # istersen p>eps_state filtrele
                        out[(i, j, 1)] = p
        
                tail = total_ge1 - mass_listed
                if tail < eps:
                    return out, j, tail  # j: son yazılan level
        
                pij = pij @ R
        
            raise RuntimeError("Reached Jmax_cap before tail mass dropped below eps. Check stability or increase cap.")
    
        pi0, pi1, R, empty = solve_boundary_and_pi1(L1, L2, h2, mu, n_star)
        states2, J_used, tail = list_states_until_tail_mass(pi0, pi1, R, n_star, empty, eps=1e-10)

        return states2

    if n_star == 0:
        import numpy as np
        def build_blocks_nstar0(L1, L2, h2, mu):
            lam1 = float(L1)
            lam2 = float(L2) * float(h2)
        
            # Phases in levels j>=1: [S1, S2]  (size m=2)
            m = 2
        
            # A2: j -> j+1 due to type2 arrival (same phase)
            A2 = lam2 * np.eye(m)
        
            # A0: j -> j-1 due to service completion when there is waiting type2
            # From S1 at level j>=1: completion moves to S2 and consumes one waiting -> level j-1
            # From S2 at level j>=1: completion stays S2 and consumes one waiting -> level j-1
            A0 = np.zeros((m, m))
            A0[0, 1] = mu  # S1 -> S2
            A0[1, 1] = mu  # S2 -> S2
        
            # A1: same level transitions (none except diagonals)
            # In levels j>=1, only events are type2 arrivals (up) and service completions (down)
            A1 = np.zeros((m, m))
            A1[0, 0] = -(lam2 + mu)
            A1[1, 1] = -(lam2 + mu)
        
            return A0, A1, A2, lam1, lam2
        
        def solve_R_iter(A0, A1, A2, tol=1e-12, max_iter=5000):
            m = A1.shape[0]
            R = np.zeros((m, m))
            for _ in range(max_iter):
                M = A1 + R @ A0
                R_next = np.linalg.solve(M.T, (-A2).T).T  # R_next = -A2 * inv(M)
                if np.max(np.abs(R_next - R)) < tol:
                    return R_next
                R = R_next
            raise RuntimeError("R did not converge; check stability or use a more robust R-solver.")
        
        import numpy as np

        def solve_stationary_nstar0_as_ijk(L1, L2, h2, mu, eps=1e-12, Jmax_cap=1_000_000, tol=1e-12):
            """
            n_star=0 exact solution, returned as dict with (i,j,k) keys.
        
            Convention (n_star=0 only):
              (0,0,0) : idle
              (1,j,1) : busy, serving type1, with j type2 waiting
              (0,j,1) : busy, serving type2, with j type2 waiting
            Here i is a PHASE indicator (0 or 1). j is #type2 waiting. k in {0,1}.
            """
            lam1 = float(L1)
            lam2 = float(L2) * float(h2)
            mu = float(mu)
        
            # QBD levels j>=1, phases: [S1, S2] where
            # phase0 = S1 (serving type1), phase1 = S2 (serving type2)
            m = 2
        
            # A2: up due to type2 arrival (same phase)
            A2 = lam2 * np.eye(m)
        
            # A0: down due to service completion consuming one waiting type2
            # from S1 -> S2, from S2 -> S2
            A0 = np.zeros((m, m))
            A0[0, 1] = mu
            A0[1, 1] = mu
        
            # A1: same level (only diagonals)
            A1 = np.zeros((m, m))
            A1[0, 0] = -(lam2 + mu)
            A1[1, 1] = -(lam2 + mu)
        
            # solve R via iteration: R = -A2 (A1 + R A0)^(-1)
            R = np.zeros((m, m))
            for _ in range(5000):
                M = A1 + R @ A0
                R_next = np.linalg.solve(M.T, (-A2).T).T
                if np.max(np.abs(R_next - R)) < tol:
                    R = R_next
                    break
                R = R_next
            else:
                raise RuntimeError("R did not converge (n_star=0). Check stability or use stronger R-solver.")
        
            # Boundary j=0 states: [E, S1_0, S2_0]
            # E = idle
            # S1_0 = serving type1, no waiting type2 (j=0)
            # S2_0 = serving type2, no waiting type2 (j=0)
            m0 = 3
            E, S1_0, S2_0 = 0, 1, 2
        
            B00 = np.zeros((m0, m0))
            B01 = np.zeros((m0, m))   # to level1 phases [S1,S2]
            B10 = np.zeros((m, m0))   # from level1 to level0
        
            # E -> S1_0 via type1 arrival; E -> S2_0 via type2 arrival
            B00[E, S1_0] = lam1
            B00[E, S2_0] = lam2
        
            # completion at j=0 returns to idle
            B00[S1_0, E] = mu
            B00[S2_0, E] = mu
        
            # type2 arrival while busy at j=0 creates queue (j:0->1), phase preserved
            B01[S1_0, 0] = lam2  # -> level1 phase S1
            B01[S2_0, 1] = lam2  # -> level1 phase S2
        
            # from level1 down to level0 via completion consuming waiting type2:
            # (S1, j=1)->(S2, j=0), (S2, j=1)->(S2, j=0)
            B10[0, S2_0] = mu
            B10[1, S2_0] = mu
        
            # diagonals for B00 (include out to level1)
            for s in range(m0):
                out = np.sum(B00[s, :]) + np.sum(B01[s, :])
                B00[s, s] = -out
        
            # Solve for [pi0, pi1] with boundary equations + normalization
            dim = m0 + m
            G = np.zeros((dim, dim))
            G[:m0, :m0] = B00
            G[:m0, m0:] = B01
            G[m0:, :m0] = B10
            G[m0:, m0:] = A1 + R @ A0
        
            I = np.eye(m)
            ones0 = np.ones(m0)
            ones1 = np.ones(m)
            v = np.linalg.solve(I - R, ones1)  # (I-R)^(-1) 1
        
            norm_row = np.concatenate([ones0, v])
        
            Msys = G.T.copy()
            b = np.zeros(dim)
            Msys[-1, :] = norm_row
            b[-1] = 1.0
        
            x = np.linalg.solve(Msys, b)
            pi0 = x[:m0]   # [E, S1_0, S2_0]
            pi1 = x[m0:]   # [S1 at j=1, S2 at j=1]
        
            # Now materialize states as (i,j,k) dict using tail-mass stopping
            # total mass in j>=1:
            total_ge1 = float(pi1 @ np.linalg.solve(I - R, ones1))
        
            states = {}
            # idle
            states[(0, 0, 0)] = float(pi0[E])
        
            # j=0 busy states
            states[(1, 0, 1)] = float(pi0[S1_0])  # serving type1
            states[(0, 0, 1)] = float(pi0[S2_0])  # serving type2
        
            pij = pi1.copy()  # j=1
            mass_listed = 0.0
            for j in range(1, Jmax_cap + 1):
                level_mass = float(np.sum(pij))
                mass_listed += level_mass
        
                # phase0=S1 -> encode as i=1
                p_s1 = float(pij[0])
                if p_s1 >= eps:
                    states[(1, j, 1)] = p_s1
        
                # phase1=S2 -> encode as i=0
                p_s2 = float(pij[1])
                if p_s2 >= eps:
                    states[(0, j, 1)] = p_s2
        
                tail = total_ge1 - mass_listed
                if tail < eps:
                    break
        
                pij = pij @ R
            else:
                raise RuntimeError("Reached Jmax_cap before tail mass dropped below eps (n_star=0).")
        
            return states, (pi0, pi1, R)

        
        def list_states_until_tail_mass_nstar0(pi0, pi1, R, eps=1e-12, Jmax_cap=1_000_000):
            # Report states as (phase, j) with phase in {"S1","S2"} and plus "E"
            out = {"E": float(pi0[0]), ("S1", 0): float(pi0[1]), ("S2", 0): float(pi0[2])}
        
            m = 2
            I = np.eye(m)
            ones = np.ones(m)
        
            total_ge1 = float(pi1 @ np.linalg.solve(I - R, ones))
        
            pij = pi1.copy()  # j=1 row vector over [S1,S2]
            mass_listed = 0.0
        
            for j in range(1, Jmax_cap + 1):
                level_mass = float(np.sum(pij))
                mass_listed += level_mass
        
                out[("S1", j)] = float(pij[0])
                out[("S2", j)] = float(pij[1])
        
                tail = total_ge1 - mass_listed
                if tail < eps:
                    return out, j, tail
        
                pij = pij @ R
        
            raise RuntimeError("Reached Jmax_cap before tail mass dropped below eps. Check stability or increase cap.")

        states, aux = solve_stationary_nstar0_as_ijk(L1, L2, h2, mu, eps=1e-10)
        return states




In [3]:
def wait_time_t2(i, j, k, L1, L2, mu, R1, R2, P, c1, c2, n_star):
    p = L1 / mu
    n = i + k
    K = n_star + 1

    t1 = (1 / mu) * (1 / (1 - p) ** 2) * (n * (1 - p) - (1 - p ** n) * p ** (K - n + 1))

    m = 1
    t2 = (1 / mu) * (1 / (1 - p) ** 2) * (m * (1 - p) - (1 - p ** m) * p ** (K - m + 1))

    return t1 + t2 * j

In [4]:
def build_decision_set(L1, L2, R1, c1, R2, c2, P, mu):
    dec_list = []
    n_star = math.floor(mu * ((R1-P) / c1))

    for n in range(0, n_star+1):
        h = 0
        while h <= 1:
            dec_list.append((n,h))
            h += 0.01
        if (n,1) not in dec_list:
            dec_list.append((n,1))

    return dec_list
            

In [5]:
def utility(L1, L2, R1, c1, R2, c2, P, mu, n, h2):
    # TYPE 1
    n_star = n
    type1_probs = {}
    π0 = (mu - L1) / (mu + L2 - (L1 + L2) * ((L1 / mu) ** (n_star+1)))
    type1_probs[0] = π0
    for n in range(1, n_star + 2):
        π_n = π0 * ((L1 / mu) ** (n - 1)) * ((L1 + L2) / mu)
        type1_probs[n] = π_n
    result = type1_probs

    #Type-1 Joining rate and equilibrium lambda
    eq_L1 = L1*(1-result[(n_star+1)])
    if P>R1:
        eq_L1=0

    h1_star = 1-result[(n_star+1)]
    state1 = list(result.keys())
    
    U1 = 0
    W1 = 0
    for s in state1:
        if s <= n_star:
            W1 +=result[s]*s/mu
            U1 += result[s]*(R1 - c1*s/mu)


    # TYPE 2
    res2 = solution_for_h2(L1, L2, R1, c1, R2, c2, P, mu, n, h2)
    state2 = list(res2.keys())
    U2 = 0
    W2 = 0
    for s in state2:
        i,j,k = s[0], s[1], s[2]
        next = (i, j+1, k)
        if next in state2:
            W2 += res2[s]*wait_time_t2(i, j, k, L1, L2, mu, R1, R2, P, c1, c2, n_star)
            U2 += res2[s]*(R2-c2*wait_time_t2(i, j, k, L1, L2, mu, R1, R2, P, c1, c2, n_star))

    return h1_star, h2, U1, U2, W1, W2
    
    

In [6]:
def max_util(L1, L2, R1, c1, R2, c2, P, mu):
    decision_set = build_decision_set(L1, L2, R1, c1, R2, c2, P, mu)
    results = {}

    best_value = float("-inf")
    best_decision = None
    
    for dec in decision_set:
        nn, hh = dec
        h1, h2, U1, U2, W1, W2 = utility(L1, L2, R1, c1, R2, c2, P, mu, nn, hh)
        U = L1*h1*U1 + L2*h2*U2
        results[dec] = U

        if U > best_value:
            best_value = U
            best_decision = dec
            best_h = h1, h2
            best_W = W1, W2

    return best_decision, best_value, best_h, best_W

In [7]:
max_util(1.25, 3.75, 3, 9, 50, 0.9, 0, 10)

((2, 1),
 190.47019086851614,
 (0.9950248756218906, 1),
 (0.03980099502487562, 0.09351670775350938))